# 📘 Módulo 01: DDL & DML Básicos en Cartera de Crédito
### 💳 Modelamiento Entidad-Relación (1:N), Definición y Manipulación de Datos en Entornos Financieros
**Asignatura:** Bases de Datos Avanzadas  
**Entorno:** Google Colab / DuckDB (100% In-Memory)  
---
## 🎯 Objetivos de Aprendizaje
1. Dominar los comandos **DDL (Data Definition Language)**: `CREATE TABLE`, `ALTER TABLE`, `DROP TABLE`, `TRUNCATE TABLE` y Restricciones de Integridad (`PK`, `FK`, `UNIQUE`, `NOT NULL`, `CHECK`, `DEFAULT`).
2. Dominar los comandos **DML (Data Manipulation Language)**: `INSERT INTO`, `UPDATE`, `DELETE`, `SELECT`.
3. Comprender el **Diagrama Entidad-Relación (MER/DER)** aplicado a un sistema real de **Cartera de Crédito y Cobranzas**.
4. Analizar las cardinalidades relacionales **1:N (Uno a Muchos)**: `Clientes` $\rightarrow$ `Créditos`, `Créditos` $\rightarrow$ `Pagos` y `Créditos` $\rightarrow$ `Garantías`.
5. Comparar los dialectos y sintaxis especiales en la **Matriz Multi-Motor Empresarial** (SQL Server, Informix, PostgreSQL, Oracle, MariaDB, DuckDB).

In [1]:
# ==============================================================================
# 🚀 SETUP INICIAL AUTO-CONTENIDO: DUCKDB & GOOGLE COLAB (100% IN-MEMORY)
# ==============================================================================
!pip install duckdb duckdb-engine jupysql -q

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Conectar a motor DuckDB en memoria
conn = duckdb.connect(database=':memory:')

%sql conn
print('✅ Entorno DuckDB inicializado para el Módulo 01 (Cartera de Crédito).')

✅ Entorno DuckDB inicializado para el Módulo 01 (Cartera de Crédito).


## 1. Diagrama Entidad-Relación (MER/DER) del Sistema de Cartera
Para gestionar una **Cartera de Créditos**, debemos estructurar las entidades financieras, sus atributos y sus cardinalidades:
```text
  ┌──────────────────┐ 1        N ┌──────────────────┐ 1        N ┌──────────────────┐
  │ CLIENTES_CARTERA │────────────│     CREDITOS     │────────────│  PAGOS_CARTERA   │
  ├──────────────────┤            ├──────────────────┤            ├──────────────────┤
  │ PK cliente_id    │            │ PK credito_id    │            │ PK pago_id       │
  │    nombre        │            │ FK cliente_id    │            │ FK credito_id    │
  │    cupo_aprobado │            │    monto, saldo  │            │    monto_pagado  │
  └──────────────────┘            └────────┬─────────┘            └──────────────────┘
                                           │ 1
                                           │
                                           │ N
                                  ┌──────────────────┐
                                  │    GARANTIAS     │
                                  ├──────────────────┤
                                  │ PK garantia_id   │
                                  │ FK credito_id    │
                                  │ tipo, valor      │
                                  └──────────────────┘
```
### 📐 Explicación Conceptual de Cardinalidades:
1. **Relación 1:N (Uno a Muchos):**
   - **`CLIENTES_CARTERA` $\rightarrow$ `CREDITOS`:** Un único cliente puede solicitar múltiples desembolsos de crédito ($N$). La clave primaria `cliente_id` migra como **Clave Foránea (`FK`)** a la tabla `CREDITOS`.
   - **`CREDITOS` $\rightarrow$ `PAGOS_CARTERA`:** Un crédito puede recibir múltiples abonos a capital o intereses ($N$) a lo largo del tiempo. La clave `credito_id` migra como **`FK`** a `PAGOS_CARTERA`.
   - **`CREDITOS` $\rightarrow$ `GARANTIAS`:** Un crédito puede tener varias garantías (por ejemplo, hipoteca y vehículo), pero cada registro de garantía pertenece a un solo crédito. Por eso, `credito_id` es una **`FK`** en `GARANTIAS`.

Este ejemplo usa una relación **1:N** para mantener el modelo básico. Si una misma garantía tuviera que respaldar varios créditos, la relación sería **N:M** y se necesitaría una tabla asociativa `credito_garantias`.

## 2. Matriz Comparativa Empresarial de Dialectos SQL
| Concepto / Tipo | Microsoft SQL Server / Azure SQL | IBM Informix (IDS) | PostgreSQL | Oracle Database | MariaDB / MySQL | DuckDB (Google Colab) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Autoincremental** | `IDENTITY(1,1)` | `SERIAL` / `SERIAL8` | `GENERATED ALWAYS AS IDENTITY` | `IDENTITY` / `SEQUENCE` | `AUTO_INCREMENT` | `BIGINT` / `SEQUENCE` |
| **Texto / Cadenas** | `VARCHAR(n)` / `NVARCHAR(n)` | `VARCHAR(n)` / `LVARCHAR` | `VARCHAR(n)` / `TEXT` | `VARCHAR2(n)` | `VARCHAR(n)` / `TEXT` | `VARCHAR` |
| **Moneda / Decimal** | `DECIMAL(p,s)` / `MONEY` | `MONEY(p,s)` / `DECIMAL` | `NUMERIC(p,s)` / `DECIMAL` | `NUMBER(p,s)` | `DECIMAL(p,s)` | `DECIMAL(p,s)` |
| **Fecha & Hora** | `DATETIME2` / `DATETIMEOFFSET` | `DATETIME YEAR TO SECOND` | `TIMESTAMP` / `TIMESTAMPTZ` | `DATE` / `TIMESTAMP` | `DATETIME` / `TIMESTAMP` | `TIMESTAMP` |
| **Documentos JSON** | `JSON` / `OPENJSON()` | `JSON` / `BSON` | `JSONB` / `JSON` | `JSON` / `CLOB` | `JSON` | `JSON` |
| **Upsert (Modificar/Insertar)** | `MERGE INTO ...` | `MERGE INTO ...` | `INSERT ... ON CONFLICT` | `MERGE INTO ...` | `INSERT ... ON DUPLICATE KEY` | `INSERT ... ON CONFLICT` |
| **Diagnóstico Plan** | `SET SHOWPLAN_TEXT ON;` | `SET EXPLAIN ON;` | `EXPLAIN ANALYZE ...;` | `EXPLAIN PLAN FOR ...;` | `EXPLAIN FORMAT=JSON ...;` | `EXPLAIN ...;` |

## 3. Comandos DDL: Creación del Esquema de Cartera (CREATE TABLE)
A continuación crearemos las 4 tablas del dominio de Cartera aplicando restricciones de integridad de dominio, entidad y referencial.

### ¿Qué hace DDL? Palabras clave y restricciones del esquema

**DDL** (*Data Definition Language*, Lenguaje de Definición de Datos) define la estructura de la base de datos: crea o modifica tablas y sus columnas. Aquí usamos `CREATE TABLE` para definir `clientes_cartera` y `creditos` antes de insertar registros.

- `CREATE TABLE nombre (...)`: crea una tabla y enumera sus columnas, tipos y restricciones.
- `IF NOT EXISTS`: crea la tabla solo si todavía no existe. **No actualiza** la definición de una tabla que ya existe.
- `INT`, `VARCHAR`, `DECIMAL(15,2)` y `DATE`: tipos de datos para enteros, texto, números decimales y fechas. En `DECIMAL(15,2)`, se permiten 15 dígitos en total, 2 de ellos después del punto decimal.

**Constraints** (restricciones) son reglas que el motor de base de datos valida al insertar o modificar datos. Protegen la integridad y ayudan a evitar valores inválidos:

- `PRIMARY KEY`: identifica cada fila de forma única; no admite valores duplicados ni `NULL`.
- `NOT NULL`: exige que la columna tenga un valor.
- `UNIQUE`: impide repetir un valor, como el documento de identidad.
- `CHECK (condición)`: acepta solo valores que cumplan la condición; por ejemplo, un score entre 300 y 950 o un monto mayor que cero.
- `DEFAULT valor`: asigna un valor cuando se omite la columna en el `INSERT`.
- `REFERENCES tabla(columna)`: crea una clave foránea. En `creditos`, `cliente_id REFERENCES clientes_cartera(cliente_id)` obliga a que cada crédito apunte a un cliente existente; esta FK implementa la relación 1:N.

Si una operación viola una constraint (por ejemplo, repetir una PK, insertar un score fuera del rango o referenciar un cliente inexistente), el motor rechaza la operación y reporta un error de restricción. Las reglas no corrigen el dato automáticamente: hay que revisar y corregir el valor o la relación.

In [ ]:
%%sql
-- 1. Tabla de Clientes de Cartera
CREATE TABLE IF NOT EXISTS clientes_cartera (
    cliente_id INT PRIMARY KEY,
    nombre VARCHAR NOT NULL,
    doc_identidad VARCHAR UNIQUE NOT NULL,
    score_credito INT CHECK (score_credito BETWEEN 300 AND 950),
    cupo_aprobado DECIMAL(15,2) DEFAULT 5000000.00,
    ciudad VARCHAR DEFAULT 'Bogotá'
);

-- 2. Tabla de Créditos (Relación 1:N con Clientes)
CREATE TABLE IF NOT EXISTS creditos (
    credito_id INT PRIMARY KEY,
    cliente_id INT NOT NULL REFERENCES clientes_cartera(cliente_id),
    monto_desembolsado DECIMAL(15,2) CHECK (monto_desembolsado > 0),
    saldo_actual DECIMAL(15,2) CHECK (saldo_actual >= 0),
    tasa_interes_anual DECIMAL(5,2) CHECK (tasa_interes_anual >= 0),
    fecha_desembolso DATE NOT NULL,
    estado VARCHAR DEFAULT 'VIGENTE' CHECK (estado IN ('VIGENTE', 'MORA', 'PAGADO', 'CASTIGADO'))
);

-- 3. Tabla de Pagos de Cartera (Relación 1:N con Créditos)
CREATE TABLE IF NOT EXISTS pagos_cartera (
    pago_id INT PRIMARY KEY,
    credito_id INT NOT NULL REFERENCES creditos(credito_id),
    fecha_pago TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    monto_pagado DECIMAL(15,2) CHECK (monto_pagado > 0),
    metodo_pago VARCHAR DEFAULT 'PSE'
);

-- 4. Tabla de Garantías
CREATE TABLE IF NOT EXISTS garantias (
    garantia_id INT PRIMARY KEY,
    credito_id INT NOT NULL REFERENCES creditos(credito_id),
    tipo_garantia VARCHAR NOT NULL,
    valor_comercial DECIMAL(15,2) CHECK (valor_comercial > 0),
    descripcion VARCHAR
);

## 4. Modificación de Esquema con DDL (ALTER TABLE, TRUNCATE, DROP)
El comando `ALTER TABLE` permite modificar estructuras existentes sin perder datos.

In [ ]:
%%sql
-- Modificar tabla clientes_cartera: Agregar columna de email
ALTER TABLE clientes_cartera ADD COLUMN email VARCHAR;

-- Verificar la estructura modificada
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'clientes_cartera';

## 5. DML: Manipulación de los datos y sintaxis básica

**DML** (*Data Manipulation Language*, Lenguaje de Manipulación de Datos) permite insertar, modificar y eliminar registros almacenados en las tablas. En este módulo también usaremos `SELECT` para consultar los datos; en algunas clasificaciones, las consultas se consideran **DQL** (*Data Query Language*).

### `INSERT`: insertar registros

Agrega una o varias filas. Las columnas indicadas deben corresponder, en cantidad y orden, con los valores proporcionados. Las columnas omitidas deben admitir `NULL`, tener un `DEFAULT` o ser autogeneradas.

```sql
INSERT INTO nombre_tabla (columna1, columna2)
VALUES (valor1, valor2);
```

### `UPDATE`: modificar registros

Cambia los valores de las filas que cumplen la condición. Incluye `WHERE` para limitar cuáles registros se actualizan; **si se omite, se actualizarán todas las filas**.

```sql
UPDATE nombre_tabla
SET columna1 = nuevo_valor
WHERE condicion;
```

### `DELETE`: eliminar registros

Elimina las filas que cumplen la condición. Incluye `WHERE` para limitar la eliminación; **si se omite, se eliminarán todas las filas** (la tabla permanece).

```sql
DELETE FROM nombre_tabla
WHERE condicion;
```

### `SELECT`: consultar registros

Recupera las columnas y filas solicitadas. `WHERE` filtra los resultados y `ORDER BY` define el orden de presentación.

```sql
SELECT columna1, columna2
FROM nombre_tabla
WHERE condicion
ORDER BY columna1;
```

En las siguientes celdas aplicaremos estas instrucciones a clientes, créditos, pagos y garantías.

## 6. Comandos DML: Poblar la Cartera (INSERT INTO)
Insertamos datos realistas para probar las relaciones 1:N entre clientes y créditos, créditos y pagos, y créditos y garantías.

In [ ]:
%%sql
-- 1. Insertar Clientes
INSERT INTO clientes_cartera (cliente_id, nombre, doc_identidad, score_credito, cupo_aprobado, ciudad, email) VALUES
(1, 'Inversiones Empresariales Bogotá S.A.S.', '900123456-1', 820, 150000000.00, 'Bogotá', 'contacto@inversionesbogota.co'),
(2, 'Construcciones del Valle Ltda.', '890987654-3', 680, 80000000.00, 'Cali', 'gerencia@convalle.com'),
(3, 'María Fernanda Gómez', '1018456789', 750, 25000000.00, 'Medellín', 'mfgomez@email.com'),
(4, 'Comercializadora Caribe S.A.', '800555444-2', 590, 45000000.00, 'Barranquilla', 'ventas@caribesa.co'),
(5, 'Jorge Enrique Restrepo', '79456123', 420, 10000000.00, 'Bogotá', 'jrestrepo@email.com');

-- 2. Insertar Créditos (Relación 1:N)
INSERT INTO creditos (credito_id, cliente_id, monto_desembolsado, saldo_actual, tasa_interes_anual, fecha_desembolso, estado) VALUES
(101, 1, 100000000.00, 75000000.00, 16.50, '2023-01-15', 'VIGENTE'),
(102, 1, 40000000.00, 38000000.00, 18.00, '2023-06-10', 'VIGENTE'),
(103, 2, 60000000.00, 60000000.00, 19.50, '2023-09-01', 'MORA'),
(104, 3, 15000000.00, 0.00, 15.00, '2022-11-20', 'PAGADO'),
(105, 4, 30000000.00, 32500000.00, 22.00, '2023-03-12', 'MORA'),
(106, 5, 8000000.00, 8900000.00, 24.00, '2023-02-05', 'CASTIGADO');

-- 3. Insertar Garantías (cada fila pertenece a un crédito)
INSERT INTO garantias (garantia_id, credito_id, tipo_garantia, valor_comercial, descripcion) VALUES
(501, 101, 'Hipotecaria', 250000000.00, 'Bodega Industrial Zona Franca Fontibón'),
(502, 101, 'Vehicular', 85000000.00, 'Camión Chevrolet NPR 2022'),
(503, 102, 'Vehicular', 85000000.00, 'Vehículo utilitario asociado al crédito 102'),
(504, 103, 'Fiador Solvente', 50000000.00, 'Fiador DSI con propiedad raíz'),
(505, 105, 'Pagaré firmado', 10000000.00, 'Pagaré a la orden con firma de representante legal'),
(506, 106, 'Vehicular', 42000000.00, 'Vehículo de respaldo del crédito 106');

-- 4. Insertar Pagos de Cartera (Relación 1:N)
INSERT INTO pagos_cartera (pago_id, credito_id, fecha_pago, monto_pagado, metodo_pago) VALUES
(9001, 101, '2023-02-15 10:30:00', 12500000.00, 'Transferencia Bancaria'),
(9002, 101, '2023-03-15 14:20:00', 12500000.00, 'Transferencia Bancaria'),
(9003, 104, '2023-05-10 09:15:00', 15000000.00, 'PSE'),
(9004, 102, '2023-07-10 16:45:00', 2000000.00, 'Ventanilla');

## 7. Comandos DML: Consultas y Actualizaciones Operativas (SELECT, UPDATE, DELETE)
### 7.1 Consulta del Estado de la Cartera (SELECT)

In [ ]:
%%sql
-- Visualizar cartera consolidada con datos del cliente y saldos
SELECT 
    c.cliente_id,
    c.nombre AS cliente,
    cr.credito_id,
    cr.monto_desembolsado,
    cr.saldo_actual,
    cr.estado,
    c.score_credito
FROM clientes_cartera c
JOIN creditos cr ON c.cliente_id = cr.cliente_id
ORDER BY cr.saldo_actual DESC;

-- Verificar la relación 1:N entre créditos y garantías
SELECT cr.credito_id, g.garantia_id, g.tipo_garantia, g.valor_comercial
FROM creditos cr
JOIN garantias g ON cr.credito_id = g.credito_id
ORDER BY cr.credito_id, g.garantia_id;

### 7.2 Actualización Masiva de Cartera en Mora (UPDATE)
En la gestión financiera, debemos aplicar penalizaciones por mora e incrementar el saldo en función de los recargos.

In [ ]:
%%sql
-- Aplicar un recargo de mora del 5% a todos los créditos en estado 'MORA'
UPDATE creditos
SET saldo_actual = saldo_actual * 1.05
WHERE estado = 'MORA';

-- Verificar saldos incrementados en créditos en mora
SELECT credito_id, cliente_id, saldo_actual, estado 
FROM creditos 
WHERE estado = 'MORA';

### 7.3 Eliminación y Control de Integridad Referencial (DELETE)
¿Qué ocurre si intentamos eliminar un cliente que tiene créditos activos registrados?
Al ejecutar el comando `DELETE` sobre una tupla padre que posee claves foráneas asociadas en tablas hijas, el motor **DuckDB bloquea la acción** preservando la Integridad Referencial.

In [ ]:
%%sql
-- 1. INTENTO DE VIOLACIÓN DE FK (Bloqueado por el Motor SQL):
-- Intentar eliminar el cliente_id = 1 (posee los créditos 101 y 102 en la tabla creditos).
-- Al ejecutar esta celda, DuckDB arrojará: 'Constraint Error: Violates foreign key constraint'
DELETE FROM clientes_cartera WHERE cliente_id = 1;

### 7.4 Eliminación Correcta Respetando el Orden Referencial (DELETE)
Para borrar exitosamente un registro con dependencias sin violar las claves foráneas, debemos eliminar primero los registros hijos o desasociar sus claves.

In [ ]:
%%sql
-- 2. ELIMINACIÓN VÁLIDA EN ORDEN CORRECTO:
-- Paso A: Eliminar primero la garantía hija asociada al crédito 106
DELETE FROM garantias WHERE credito_id = 106;

-- Paso B: Eliminar el crédito 106 en la tabla creditos
DELETE FROM creditos WHERE credito_id = 106;

-- Paso C: Ahora es posible eliminar al cliente 5 sin violar la FK
DELETE FROM clientes_cartera WHERE cliente_id = 5;

-- Verificar eliminación exitosa del cliente 5
SELECT * FROM clientes_cartera WHERE cliente_id = 5;

## 8. Retos Prácticos de Cartera para el Estudiante 🏋️‍♂️
### ✏️ Reto 1: DDL - Crear la tabla `gestiones_cobranza` (Relación 1:N)
Crea una nueva tabla llamada `gestiones_cobranza` con los siguientes campos:
- `gestion_id` INT PRIMARY KEY
- `credito_id` INT REFERENCES creditos(credito_id)
- `fecha_gestion` TIMESTAMP DEFAULT CURRENT_TIMESTAMP
- `tipo_contacto` VARCHAR CHECK (tipo_contacto IN ('Llamada', 'Correo', 'Visita Campo', 'WhatsApp'))
- `compromiso_pago` DATE
- `observaciones` VARCHAR
### ✏️ Reto 2: DML - Insertar 2 gestiones de cobranza para el crédito 103 (en MORA)
### ✏️ Reto 3: DML UPDATE - Actualizar el Score de Crédito
Incrementa en +50 puntos el `score_credito` de los clientes que tengan al menos un crédito en estado `'PAGADO'`.

In [ ]:
%%sql
-- 🛠 Espacio para la solución del estudiante: Escribe aquí tus instrucciones DDL y DML

-- Reto 1: CREATE TABLE gestiones_cobranza ...

-- Reto 2: INSERT INTO gestiones_cobranza ...

-- Reto 3: UPDATE clientes_cartera ...

In [ ]:
%%sql
-- ==============================================================================
-- 🛠 SOLUCIÓN A LOS RETOS PRÁCTICOS DE CARTERA (MÓDULO 01)
-- ==============================================================================

-- ------------------------------------------------------------------------------
-- ✏️ RETO 1: DDL - Crear la tabla `gestiones_cobranza` (Relación 1:N con Créditos)
-- ------------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS gestiones_cobranza (
    gestion_id INT PRIMARY KEY,
    credito_id INT NOT NULL REFERENCES creditos(credito_id),
    fecha_gestion TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    tipo_contacto VARCHAR CHECK (tipo_contacto IN ('Llamada', 'Correo', 'Visita Campo', 'WhatsApp')),
    compromiso_pago DATE,
    observaciones VARCHAR
);


-- ------------------------------------------------------------------------------
-- ✏️ RETO 2: DML - Insertar 2 gestiones de cobranza para el crédito 103 (en MORA)
-- ------------------------------------------------------------------------------
INSERT INTO gestiones_cobranza (gestion_id, credito_id, fecha_gestion, tipo_contacto, compromiso_pago, observaciones) VALUES
(7001, 103, '2024-03-01 09:30:00', 'Llamada', '2024-03-15', 'Cliente solicita plazo adicional. Se acuerda compromiso de pago parcial.'),
(7002, 103, '2024-03-10 14:15:00', 'WhatsApp', '2024-03-20', 'Recordatorio enviado vía WhatsApp. Representante legal confirma transferencia.');


-- ------------------------------------------------------------------------------
-- ✏️ RETO 3: DML UPDATE - Actualizar el Score de Crédito (+50 puntos a clientes puntuales)
-- ------------------------------------------------------------------------------
UPDATE clientes_cartera
SET score_credito = LEAST(score_credito + 50, 950)
WHERE cliente_id IN (
    SELECT DISTINCT cliente_id 
    FROM creditos 
    WHERE estado = 'PAGADO'
);


-- ------------------------------------------------------------------------------
-- 🔍 VERIFICACIÓN DE RESULTADOS
-- ------------------------------------------------------------------------------
-- 1. Consultar el score de crédito actualizado (María Fernanda Gómez sube de 750 a 800)
SELECT cliente_id, nombre, score_credito 
FROM clientes_cartera 
WHERE cliente_id IN (SELECT cliente_id FROM creditos WHERE estado = 'PAGADO');

-- 2. Consultar el historial de gestiones de cobranza registradas
SELECT * FROM gestiones_cobranza;
